# Gold template: Type 2 dimension (Fabric)

Builds a dimension that keeps history: when a tracked attribute changes, the current version is closed
and a new version is added. Facts point at the version that was current on the fact's date.

**Standards this notebook follows**

1. **Two keys.** `<name>_sk` is the version key that facts join on. The version key is issued here as `MAX(key) + ROW_NUMBER()` (Fabric lakehouse Delta tables have no identity columns), as an `INT`.
   `<name>_dk`, the durable key, is the Silver key from the key table: the same for every version of a row.
2. **Validity.** `valid_from` (inclusive) and `valid_to` (exclusive), as dates, plus `is_current`. The first
   version starts at `1900-01-01` so older facts still find a version; the current one ends at `9999-12-31`.
   A fact on day `d` takes the version where `valid_from <= d < valid_to`.
3. **Tracked and overwritten attributes.** A change in `type2_columns` makes a new version. A change in
   `type1_columns` (corrections) overwrites every version.
4. **One atomic MERGE** closes changed versions and adds new ones, so a failed run never leaves a row with no
   current version, and a re-run with no changes does nothing.
5. **Unknown member** `-1`, valid for all dates, with `Unknown` in its text columns.
6. **Audit columns:** `_type2_hash`, `_type1_hash`, `_run_id`, `_loaded_at`. Rows deleted in Silver are kept.
7. **Layers.** one lakehouse per layer, schema-enabled: `LH_Bronze`, `LH_Silver`, `LH_Gold`, with keys in `LH_Keys`.

## Who supports this notebook

This is an **enterprise template** (`gold-dim-type2` 1.0.0, Fabric). The line between enterprise and domain is drawn by the cells:

| Cells | Marked | Who owns them |
| --- | --- | --- |
| **Parameters** | `# kratosforge:parameters` | The **domain**: table names, keys, columns, measures. Values only, no code. |
| Every other code cell | `# kratosforge:standard` | The **enterprise**: it maintains, fixes and upgrades them. Don't edit them. |
| Markdown cells | none | Anyone: add notes freely. |

While the standard cells are unchanged and no code cell is added, the notebook **conforms** and the enterprise can
take it over: check it with `fabric-desktop templates check <notebook>`, then hand it over with
`fabric-desktop templates handover <notebook> --domain <domain> --owner <you>`.
Editing a standard cell, or adding a code cell, makes it **domain-owned** ad hoc work: it still runs, but the
domain supports it, and the check lists each change. Need logic the template doesn't have? Put it in Silver
(a derived column) or ask the enterprise to add it to the template, so every domain gets it.

In [ ]:
# kratosforge:parameters template=gold-dim-type2 platform=fabric version=1.0.0 | The domain fills in this cell: values only, no code.
# Parameters
source_table = "LH_Silver.crm.customers"
target_table = "LH_Gold.sales.dim_customer"
source_key_column = "customer_sk"    # the Silver key from the key table
sk_column = "customer_sk"            # the version key, issued here; facts join on it
durable_key_column = "customer_dk"   # the Silver key, the same for every version
business_key = {"customer_id": "int"}
type2_columns = {"segment": "string", "city": "string"}   # a change makes a new version
type1_columns = {"name": "string", "email": "string"}     # a change overwrites every version
load_date = ""                       # the date changes take effect (yyyy-mm-dd); today (UTC) when empty
run_id = ""          # the pipeline or orchestrator passes its run id; one is made when empty
full_reload = False  # True re-reads the whole source (after a logic change or a backfill)

In [ ]:
# kratosforge:standard template=gold-dim-type2 platform=fabric version=1.0.0 cell=setup | Enterprise-managed: change the Parameters cell, not this one.
import re
import time
import uuid
from datetime import datetime, timezone
from functools import reduce

from pyspark.errors import AnalysisException
from pyspark.sql import Window
from pyspark.sql import functions as F

UNKNOWN_SK = -1  # the Unknown member: rows whose key can't be found point here, never at NULL
_TABLE_PART = re.compile(r"[A-Za-z_][A-Za-z0-9_ -]{0,127}")
_COLUMN = re.compile(r"[A-Za-z_][A-Za-z0-9_]{0,127}")
_TYPE = re.compile(r"(?i)(string|boolean|tinyint|smallint|int|bigint|float|double|date|timestamp|decimal\(\d{1,2},\s*\d{1,2}\))")
_RUN_ID = re.compile(r"[A-Za-z0-9_.:-]{1,64}")


def q(name: str) -> str:
    """A table name with each part in backticks. Parameters arrive from pipelines, so they are checked."""
    parts = name.split(".")
    if not 1 <= len(parts) <= 4 or not all(_TABLE_PART.fullmatch(x) for x in parts):
        raise ValueError(f"Not a table name: {name!r}")
    return ".".join(f"`{x}`" for x in parts)


def c(name: str) -> str:
    if not _COLUMN.fullmatch(name):
        raise ValueError(f"Not a column name: {name!r}")
    return f"`{name}`"


def checked_type(dtype: str) -> str:
    if not _TYPE.fullmatch(dtype.strip()):
        raise ValueError(f"Not a supported type: {dtype!r}")
    return dtype.strip()


def table_exists(name: str) -> bool:
    try:
        spark.table(q(name))  # resolved now, so a missing table raises here
        return True
    except AnalysisException:
        return False


def row_hash(columns: list) -> "F.Column":
    """SHA-256 of the columns, NULL-safe: a change in any of them changes the hash."""
    parts = [F.coalesce(F.col(x).cast("string"), F.lit("\u0000")) for x in columns]
    return F.sha2(F.concat_ws("\u001f", *parts), 256)


run_id = run_id or str(uuid.uuid4())
if not _RUN_ID.fullmatch(run_id):
    raise ValueError("run_id may only hold letters, digits and _ . : -")
loaded_at = datetime.now(timezone.utc).replace(tzinfo=None)
print(f"Run {run_id} at {loaded_at:%Y-%m-%d %H:%M:%S} UTC")

## 1. Read Silver

In [ ]:
# kratosforge:standard template=gold-dim-type2 platform=fabric version=1.0.0 cell=step-1 | Enterprise-managed: change the Parameters cell, not this one.
attributes = {**business_key, **type2_columns, **type1_columns}
if len(attributes) != len(business_key) + len(type2_columns) + len(type1_columns):
    raise ValueError("A column can be in only one of business_key, type2_columns and type1_columns")
attributes = {c_: checked_type(t) for c_, t in attributes.items() if c(c_)}
load_date = load_date or f"{loaded_at:%Y-%m-%d}"
if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", load_date):
    raise ValueError("load_date must look like 2026-11-05")
validity = ["valid_from", "valid_to", "is_current"]
audit = ["_type2_hash", "_type1_hash", "_run_id", "_loaded_at"]

staged = (spark.table(q(source_table))
          .where(F.col(c(source_key_column)) != UNKNOWN_SK)
          .select(F.col(c(source_key_column)).cast("int").alias(durable_key_column),
                  *[F.col(c(n)).cast(t).alias(n) for n, t in attributes.items()])
          .withColumn("_type2_hash", row_hash([*business_key, *type2_columns]))
          .withColumn("_type1_hash", row_hash(list(type1_columns)))
          .withColumn("_run_id", F.lit(run_id))
          .withColumn("_loaded_at", F.lit(loaded_at).cast("timestamp")))
staged.createOrReplaceTempView("dim_staged")
def unknown_member_values(types: dict) -> list:
    """The Unknown member's value for each column: 'Unknown' for text, NULL for the rest."""
    return ["'Unknown'" if t.lower() == "string" else f"CAST(NULL AS {t})" for t in types.values()]

## 2. Create the table and its Unknown member

In [ ]:
# kratosforge:standard template=gold-dim-type2 platform=fabric version=1.0.0 cell=step-2 | Enterprise-managed: change the Parameters cell, not this one.
if not table_exists(target_table):
    ddl = ", ".join([f"{c(sk_column)} INT NOT NULL", f"{c(durable_key_column)} INT NOT NULL",
                     *[f"{c(n)} {t}" for n, t in attributes.items()],
                     "valid_from DATE NOT NULL", "valid_to DATE NOT NULL", "is_current BOOLEAN NOT NULL",
                     "`_type2_hash` STRING", "`_type1_hash` STRING", "`_run_id` STRING", "`_loaded_at` TIMESTAMP"])
    spark.sql(f"CREATE TABLE IF NOT EXISTS {q(target_table)} ({ddl}) USING DELTA")

every = [sk_column, durable_key_column, *attributes, *validity, *audit]
unknown = [str(UNKNOWN_SK), str(UNKNOWN_SK), *unknown_member_values(attributes),
           "DATE'1900-01-01'", "DATE'9999-12-31'", "true",
           "'unknown-member'", "'unknown-member'", f"'{run_id}'", f"CAST('{loaded_at:%Y-%m-%d %H:%M:%S}' AS TIMESTAMP)"]
spark.sql(f"""
MERGE INTO {q(target_table)} AS t
USING (SELECT {", ".join(f"{x} AS {c(n)}" for x, n in zip(unknown, every))}) AS s
ON t.{c(sk_column)} = s.{c(sk_column)}
WHEN NOT MATCHED THEN INSERT ({", ".join(c(n) for n in every)}) VALUES ({", ".join(f"s.{c(n)}" for n in every)})
""")

## 3. Close changed versions, add new ones, apply corrections

In [ ]:
# kratosforge:standard template=gold-dim-type2 platform=fabric version=1.0.0 cell=step-3 | Enterprise-managed: change the Parameters cell, not this one.
dk = c(durable_key_column)
# Each staged row against its current version. A new durable key inserts a first version from 1900-01-01.
# A changed tracked attribute appears twice: once with its durable key, to close the current version,
# and once with no merge key, so it inserts as the new current version from load_date.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW dim_changes AS
WITH cur AS (
    SELECT {dk}, _type2_hash FROM {q(target_table)} WHERE is_current AND {c(sk_column)} <> {UNKNOWN_SK}
)
SELECT s.{dk} AS _merge_key, s.*, cur.{dk} IS NULL AS _insert, DATE'1900-01-01' AS _valid_from
FROM dim_staged s LEFT JOIN cur ON cur.{dk} = s.{dk}
WHERE cur.{dk} IS NULL OR cur._type2_hash <> s._type2_hash
UNION ALL
SELECT NULL AS _merge_key, s.*, true AS _insert, DATE'{load_date}' AS _valid_from
FROM dim_staged s JOIN cur ON cur.{dk} = s.{dk} AND cur._type2_hash <> s._type2_hash
""")
numbered_view = "dim_numbered"
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW dim_numbered AS
SELECT ch.*,
       CASE WHEN ch._insert THEN CAST(m.max_sk + ROW_NUMBER() OVER (PARTITION BY ch._insert ORDER BY ch.{c(durable_key_column)}) AS INT) END AS _new_sk
FROM dim_changes ch
CROSS JOIN (SELECT GREATEST(COALESCE(MAX({c(sk_column)}), 0), 0) AS max_sk FROM {q(target_table)}) m
""")

result = spark.sql(f"""
MERGE INTO {q(target_table)} AS t
USING {numbered_view} AS s
ON t.{dk} = s._merge_key AND t.is_current AND t.{c(sk_column)} <> {UNKNOWN_SK}
WHEN MATCHED THEN
    UPDATE SET t.valid_to = DATE'{load_date}', t.is_current = false, t._run_id = s._run_id, t._loaded_at = s._loaded_at
WHEN NOT MATCHED THEN
    INSERT ({", ".join(c(n) for n in [sk_column, durable_key_column, *attributes, *validity, *audit])})
    VALUES ({", ".join(["s._new_sk", f"s.{c(durable_key_column)}", *[f"s.{c(n)}" for n in attributes],
            "s._valid_from", "DATE'9999-12-31'", "true", *[f"s.{c(n)}" for n in audit]])})
""").first()
print(f"Type 2 MERGE: {result.asDict() if result else 'done'}")

# Corrections to overwritten attributes go to every version of the row.
if type1_columns:
    spark.sql(f"""
    MERGE INTO {q(target_table)} AS t
    USING dim_staged AS s
    ON t.{dk} = s.{dk} AND t._type1_hash <> s._type1_hash
    WHEN MATCHED THEN UPDATE SET {", ".join(f"t.{c(n)} = s.{c(n)}" for n in [*type1_columns, "_type1_hash", "_run_id", "_loaded_at"])}
    """)

## 4. Check the result

In [ ]:
# kratosforge:standard template=gold-dim-type2 platform=fabric version=1.0.0 cell=step-4 | Enterprise-managed: change the Parameters cell, not this one.
summary = spark.sql(f"""
SELECT COUNT(*) AS row_count,
       COUNT(DISTINCT {c(sk_column)}) AS key_count,
       SUM(CASE WHEN {c(sk_column)} = {UNKNOWN_SK} THEN 1 ELSE 0 END) AS unknown_members,
       SUM(CASE WHEN valid_from > valid_to THEN 1 ELSE 0 END) AS bad_ranges
FROM {q(target_table)}
""").first()
# Exactly one current version per durable key, and versions of one key never overlap.
broken = spark.sql(f"""
SELECT {dk} FROM (
    SELECT {dk}, is_current, valid_from, valid_to,
           LEAD(valid_from) OVER (PARTITION BY {dk} ORDER BY valid_from, {c(sk_column)}) AS next_from
    FROM {q(target_table)} WHERE {c(sk_column)} <> {UNKNOWN_SK}
)
GROUP BY {dk}
HAVING SUM(CASE WHEN is_current THEN 1 ELSE 0 END) <> 1 OR SUM(CASE WHEN next_from < valid_to THEN 1 ELSE 0 END) > 0
LIMIT 5
""").take(5)
if summary["row_count"] != summary["key_count"] or summary["unknown_members"] != 1 or summary["bad_ranges"] or broken:
    raise RuntimeError(f"{target_table} breaks a Type 2 rule: {summary.asDict()}, broken durable keys {broken}")
print(f"{target_table}: {summary['row_count']} versions including the Unknown member")